# Module 095: Advanced Decorators & Metaprogramming

Phase 10: Concurrency & Internals | Duration: 2 hours

## Decorators with Arguments

In [ ]:
import functools
from typing import Any, Callable

def retry(max_attempts: int = 3) -> Callable:
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
                    print(f"Attempt {attempt + 1} failed, retrying...")
            return None
        return wrapper
    return decorator

@retry(max_attempts=3)
def unstable_operation() -> str:
    import random
    if random.random() < 0.6:
        raise ValueError("Random failure")
    return "Success!"

print(unstable_operation())

## Class Decorators

In [ ]:
import functools
from typing import Any, Type, Dict

def add_logging(cls: Type) -> Type:
    original_init = cls.__init__

    @functools.wraps(original_init)
    def new_init(self: Any, *args: Any, **kwargs: Any) -> None:
        print(f"Creating {cls.__name__} instance")
        original_init(self, *args, **kwargs)
        print(f"  Args: {args}, Kwargs: {kwargs}")

    cls.__init__ = new_init
    return cls

@add_logging
class Person:
    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age = age

p = Person("Alice", 30)
print(f"Created: {p.name}, {p.age}")

## functools.wraps in Depth

In [ ]:
import functools
from typing import Any, Callable

def bad_decorator(func: Callable) -> Callable:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        return func(*args, **kwargs)
    return wrapper

def good_decorator(func: Callable) -> Callable:
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def example1() -> int:
    """Docstring of example1."""
    return 1

@good_decorator
def example2() -> int:
    """Docstring of example2."""
    return 2

print(f"example1 name: {example1.__name__}, doc: {example1.__doc__}")
print(f"example2 name: {example2.__name__}, doc: {example2.__doc__}")

## Metaclasses

In [ ]:
from typing import Any, Dict, Type

class AutoPropertyMeta(type):
    def __new__(mcs: Type, name: str, bases: tuple, namespace: Dict[str, Any]) -> Type:
        annotations = namespace.get('__annotations__', {})
        for attr_name in annotations:
            if not attr_name.startswith('_'):
                private_name = f'_{attr_name}'

                def getter(self: Any, _name: str = attr_name, _priv: str = private_name) -> Any:
                    return getattr(self, _priv)

                def setter(self: Any, value: Any, _name: str = attr_name, _priv: str = private_name) -> None:
                    setattr(self, _priv, value)

                namespace[attr_name] = property(getter, setter)

        return super().__new__(mcs, name, bases, namespace)

class Person(metaclass=AutoPropertyMeta):
    name: str
    age: int

    def __init__(self, name: str, age: int) -> None:
        self._name = name
        self._age = age

p = Person("Bob", 25)
print(f"Name: {p.name}, Age: {p.age}")
p.name = "Robert"
print(f"Updated name: {p.name}")

## __slots__ for Memory Optimization

In [ ]:
import sys
from typing import List

class Point:
    __slots__ = ('x', 'y')

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

class RegularPoint:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

# Memory comparison
slotted: List[Point] = [Point(i, i) for i in range(1000)]
regular: List[RegularPoint] = [RegularPoint(i, i) for i in range(1000)]

print(f"Slotted instance size: {sys.getsizeof(Point(0, 0))} bytes")
print(f"Regular instance size: {sys.getsizeof(RegularPoint(0, 0))} bytes")